<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Query Pricing Demo

**Ask a question. Follow the retrieval. Inspect the cost.**

Query an existing document collection on **PageIndex Cloud** with the PageIndex SDK, stream the answer and retrieval tool calls, then inspect token usage and estimate the model cost.


1. **Set up**  Install the SDK and choose your chat model.

2. **Explore**  List the folders in the demo collection. 

3. **Ask**  Compare NVIDIA’s revenue across five years. 

4. **Measure** Inspect token usage and estimate query cost in USD. 



**Before you start:** Have your OpenAI API key ready. This demo uses an existing cloud collection, so no PDF upload or local indexing is needed.


## 01 · Set up

Install the latest PageIndex SDK.


In [ ]:
!pip install -U pageindex

### Configure your client

Set your API keys and choose `MODEL`. The client uses PageIndex Cloud for the document index and your selected model for chat.


In [ ]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"]="pageindex_demo_readonly_key"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"
MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna


client = PageIndexClient(
    index="cloud",
    chat=MODEL,
)

## 02 · Explore the collection

List the available folders in the demo collection before querying it.


In [9]:
client.list_folders(parent_folder_id="cmu3rp2g9001h0bp4t9uqramg")

{'folders': [{'id': 'cmu3rp4uo001i0bp4w067bq6g',
   'name': '2022',
   'description': 'NVIDIA fiscal year 2022',
   'parent_folder_id': 'cmu3rp2g9001h0bp4t9uqramg',
   'created_at': '2026-09-16T07:16:41',
   'updated_at': '2026-09-16T07:16:41',
   'file_count': 4,
   'children_count': 0},
  {'id': 'cmu3rq19p000y09p4negimb66',
   'name': '2023',
   'description': 'NVIDIA fiscal year 2023',
   'parent_folder_id': 'cmu3rp2g9001h0bp4t9uqramg',
   'created_at': '2026-09-16T07:17:23',
   'updated_at': '2026-09-16T07:17:23',
   'file_count': 4,
   'children_count': 0},
  {'id': 'cmu3rqfeq001l0bp4ae2x2a8p',
   'name': '2024',
   'description': 'NVIDIA fiscal year 2024',
   'parent_folder_id': 'cmu3rp2g9001h0bp4t9uqramg',
   'created_at': '2026-09-16T07:17:41',
   'updated_at': '2026-09-16T07:17:41',
   'file_count': 4,
   'children_count': 0},
  {'id': 'cmu3rqvbt000z09p47lj46i89',
   'name': '2025',
   'description': 'NVIDIA fiscal year 2025',
   'parent_folder_id': 'cmu3rp2g9001h0bp4t9uqramg'

## 03 · Ask a question

Compare NVIDIA’s revenue for the last five years. The output streams the answer alongside retrieval tool calls, with citations enabled.


In [24]:
messages = [{"role": "user", "content": "Compare NVIDIA's revenue for last five years."}]

response = None
for event in client.chat(messages, folder_id="cmu3rp2g9001h0bp4t9uqramg",
                         stream=True, citations=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]



[Tool call] get_folder_structure({"folder_id":"cmu3rp2g9001h0bp4t9uqramg","depth":2,"include_counts":true})

[Tool call] browse_documents({"folder_id":"cmu3rp4uo001i0bp4w067bq6g","recursive":false,"sort":"relevance","query":"annual revenue 10-K","limit":10})

[Tool call] browse_documents({"folder_id":"cmu3rq19p000y09p4negimb66","recursive":false,"sort":"relevance","query":"annual revenue 10-K","limit":10})

[Tool call] browse_documents({"folder_id":"cmu3rqfeq001l0bp4ae2x2a8p","recursive":false,"sort":"relevance","query":"annual revenue 10-K","limit":10})

[Tool call] browse_documents({"folder_id":"cmu3rqvbt000z09p47lj46i89","recursive":false,"sort":"relevance","query":"annual revenue 10-K","limit":10})

[Tool call] browse_documents({"folder_id":"cmu3rre91000r0cp4qb90h0fa","recursive":false,"sort":"relevance","query":"annual revenue 10-K","limit":10})

[Tool call] get_document_structure({"doc_name":"NVIDIA-FY2022-10-K.pdf","folder_id":"cmu3rp4uo001i0bp4w067bq6g","part":1})

[Tool call]

## 04 · Inspect usage & estimate cost

Review the returned token usage and estimate the model cost using the rates defined below.


In [25]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 210613,
  "input_tokens_details": {
    "cached_tokens": 104152,
    "cache_write_tokens": 106446
  },
  "output_tokens": 1883,
  "output_tokens_details": {
    "reasoning_tokens": 443
  },
  "total_tokens": 212496
}
Estimated model cost at base rates: $0.030957 USD


---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
